# Ripley's K and L: describing point patterns across distance scales

This lab compares uncorrected and translation-corrected K/L summaries for synthetic patterns. Translation correction is implemented for the rectangular study window; neither curve is a significance test without a null-model envelope.


In [ ]:
from pathlib import Path
import sys
import numpy as np
import matplotlib.pyplot as plt
ROOT = Path.cwd()
if not (ROOT / 'implementations').exists(): ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'implementations/python/ripley-k'))
from ripley_k import ripley_k


## 1. Build two synthetic patterns

The patterns share the same rectangular study window. The examples are illustrative only.

In [ ]:
rng = np.random.default_rng(41)
uniform = rng.uniform(0, 10, size=(60, 2))
centres = np.array([[2.5, 2.5], [7.5, 2.5], [2.5, 7.5], [7.5, 7.5]])
clustered = np.vstack([rng.normal(loc=c, scale=.5, size=(15, 2)) for c in centres])
clustered = np.clip(clustered, 0, 10)
patterns = {'Uniform random': uniform, 'Clustered': clustered}
fig, axes = plt.subplots(1, 2, figsize=(9, 4), sharex=True, sharey=True)
for ax, (name, points) in zip(axes, patterns.items()):
    ax.scatter(points[:, 0], points[:, 1], s=12)
    ax.set(title=name, xlim=(0, 10), ylim=(0, 10), aspect='equal'); ax.grid(True)
plt.tight_layout(); plt.show()


## 2. Compare uncorrected and translation-corrected K(r)

For a homogeneous Poisson process, theoretical K(r) is πr². Translation correction weights each pair by the inverse overlap fraction of the shifted rectangular window. The comparison is still descriptive; no significance envelope is calculated.


In [ ]:
radii = np.linspace(0, 3, 31)
fig, ax = plt.subplots(figsize=(7, 5))
for name, points in patterns.items():
    raw = ripley_k(points, bounds=(0, 10, 0, 10), radii=radii)
    corrected = ripley_k(points, bounds=(0, 10, 0, 10), radii=radii, edge_correction='translation')
    ax.plot(raw.radii, raw.k_values, label=f'{name}: uncorrected')
    ax.plot(corrected.radii, corrected.k_values, linestyle='--', label=f'{name}: translation')
ax.plot(radii, np.pi * radii**2, linestyle=':', label='CSR reference πr²')
ax.set(xlabel='Distance r', ylabel='K(r)', title='Ripley K with and without translation correction'); ax.grid(True); ax.legend(); plt.show()


## 3. Inspect the L(r) - r transformation

Under the infinite-plane homogeneous Poisson reference, L(r)=r. Compare the uncorrected and translation-corrected descriptive curves; neither is an inferential result by itself.


In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
for name, points in patterns.items():
    raw = ripley_k(points, bounds=(0, 10, 0, 10), radii=radii)
    corrected = ripley_k(points, bounds=(0, 10, 0, 10), radii=radii, edge_correction='translation')
    ax.plot(raw.radii, raw.l_values - raw.radii, label=f'{name}: uncorrected')
    ax.plot(corrected.radii, corrected.l_values - corrected.radii, linestyle='--', label=f'{name}: translation')
ax.axhline(0, linestyle=':')
ax.set(xlabel='Distance r', ylabel='L(r) - r', title='Ripley L transformation'); ax.grid(True); ax.legend(); plt.show()


## Investigate

1. At what distance scales do the patterns differ most?
2. Why is the uncorrected K estimator biased near the study-window boundary?
3. What is the difference between an edge correction and a significance envelope?
4. Why does a positive L(r)-r curve not automatically prove clustering?
5. Which additional assumptions would you need to test a point-process null model?